# Exploratory Data Analysis
## Smart Utility Expense Prediction System

This notebook performs EDA on the cleaned utility bills dataset.
- Descriptive statistics
- Missing value heatmap
- Monthly trends
- Distribution plots
- Correlation heatmap
- Seasonality / month-wise patterns

**Prerequisites:** Run `src/data_preprocessing.py` first to generate `data/processed/cleaned.csv`

In [ ]:
# Setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Paths
PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS_FIGURES = PROJECT_ROOT / "reports" / "figures"
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)

# Load cleaned data
df = pd.read_csv(DATA_PROCESSED / "cleaned.csv")
df["month_parsed"] = pd.to_datetime(df["month_parsed"].astype(str)).dt.to_period("M")

print("Dataset shape:", df.shape)
df.head()

## 1. Descriptive Statistics

In [ ]:
# Descriptive statistics
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
stats = df[numeric_cols].describe()
stats

## 2. Missing Value Heatmap

In [ ]:
# Missing value heatmap
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.isnull().T, cmap="YlOrRd", cbar_kws={"label": "Missing"}, ax=ax)
plt.title("Missing Values Heatmap")
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "missing_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Monthly Trends

In [ ]:
# Monthly trends - Line plot
df_plot = df.copy()
df_plot["month_str"] = df_plot["month_parsed"].astype(str)

fig, ax = plt.subplots(figsize=(12, 5))
for col in ["electricity", "water", "telecom", "total"]:
    if col in df.columns:
        ax.plot(df_plot["month_str"], df_plot[col], marker="o", label=col.capitalize(), markersize=4)
ax.set_xlabel("Month")
ax.set_ylabel("Amount (LKR)")
ax.set_title("Monthly Utility Expense Trends")
ax.legend()
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "monthly_trends.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Distribution Plots

In [ ]:
# Distribution plots
plot_cols = [c for c in ["electricity", "water", "telecom", "total"] if c in df.columns]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()
for i, col in enumerate(plot_cols[:4]):
    axes[i].hist(df[col].dropna(), bins=15, edgecolor="black", alpha=0.7)
    axes[i].set_title(f"{col.capitalize()} Distribution")
    axes[i].set_xlabel("Amount (LKR)")
for j in range(len(plot_cols), 4):
    axes[j].axis("off")
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Correlation Heatmap

In [ ]:
# Correlation heatmap
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f", ax=ax)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Seasonality / Month-wise Patterns

In [ ]:
# Seasonality - extract month number and aggregate
df_plot["month_num"] = pd.to_datetime(df_plot["month_parsed"].astype(str)).dt.month
monthly_avg = df_plot.groupby("month_num")[["electricity", "water", "telecom", "total"]].mean()

fig, ax = plt.subplots(figsize=(10, 5))
monthly_avg.plot(kind="bar", ax=ax)
ax.set_xlabel("Month (1=Jan, 12=Dec)")
ax.set_ylabel("Average Amount (LKR)")
ax.set_title("Seasonality: Average Expense by Month")
ax.legend(title="Utility")
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][:len(monthly_avg)])
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "seasonality.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("EDA complete. Figures saved to reports/figures/")

# Exploratory Data Analysis (EDA) - Utility Bills Dataset
## Smart Utility Expense Prediction System

This notebook performs EDA on the Sri Lankan household utility bills data (CEB, NWSDB, Telecom).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('ggplot')

PROJECT_ROOT = Path("..").resolve()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
REPORTS = PROJECT_ROOT / "reports" / "figures"
REPORTS.mkdir(parents=True, exist_ok=True)

In [ ]:
# Auto-detect and load dataset
from data_preprocessing import load_data, get_schema_report, identify_target_column

df = load_data()
df["Month"] = pd.to_datetime(df["Month"])
print("Schema & Missing Values Report:")
report = get_schema_report(df)
for k, v in report.items():
    print(f"  {k}: {v}")
print(f"\nTarget column: {identify_target_column(df)}")

## Descriptive Statistics

In [ ]:
df.describe()

## Missing Value Heatmap

In [ ]:
if df.isnull().any().any():
    plt.figure(figsize=(8, 4))
    sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
    plt.title('Missing Values Heatmap')
    plt.tight_layout()
    plt.savefig(REPORTS / 'missing_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No missing values in dataset.')

## Monthly Trends

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
numeric_cols = [c for c in df.columns if df[c].dtype in [np.float64, np.int64] and c != 'Month']
for i, col in enumerate(numeric_cols[:4]):
    ax = axes.flat[i]
    ax.plot(df['Month'], df[col], marker='o', markersize=4)
    ax.set_title(col)
    ax.set_xlabel('Month')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Monthly Trends - Utility Bills', fontsize=14)
plt.tight_layout()
plt.savefig(REPORTS / 'monthly_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## Distribution Plots

In [ ]:
numeric_cols = [c for c in df.columns if df[c].dtype in [np.float64, np.int64]]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, col in enumerate(numeric_cols[:4]):
    ax = axes.flat[i]
    df[col].hist(ax=ax, bins=15, edgecolor='black')
    ax.set_title(f'Distribution of {col}')
plt.suptitle('Distribution Plots', fontsize=14)
plt.tight_layout()
plt.savefig(REPORTS / 'distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Correlation Heatmap

In [ ]:
corr = df[numeric_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig(REPORTS / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Seasonality / Month-wise Patterns

In [ ]:
df['month_num'] = df['Month'].dt.month
monthly_avg = df.groupby('month_num')[numeric_cols].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_avg.index = [month_names[i-1] for i in monthly_avg.index]

fig, ax = plt.subplots(figsize=(12, 5))
monthly_avg.plot(kind='bar', ax=ax)
ax.set_title('Seasonality - Average Bill by Month')
ax.set_xlabel('Month')
ax.legend(bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(REPORTS / 'seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('EDA complete. Figures saved to reports/figures/')